In [1]:
import pickle
import os
import numpy as np
import pandas as pd
from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ScoreBased.GES import ges
# from causallearn.utils.DataUtils import DataUtils
from causallearn.utils.GraphUtils import GraphUtils

/Users/sahithcherumandanda/anaconda3/envs/dsc180b/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
partition_fp = os.path.join('..', 'superblue', 'superblue_18', 'metis_part_dict.pkl')
f = open(partition_fp, 'rb')
partition_dict = pickle.load(f)
f.close()

In [3]:
metis_fp = os.path.join('..', 'superblue', 'superblue_18', 'metis_part_dict.pkl')
f = open(metis_fp, 'rb')
metis_dict = pickle.load(f)
f.close()

node_features_fp = os.path.join('..', 'superblue', 'superblue_18', 'node_features.pkl')
f = open(node_features_fp, 'rb')
node_features_dict = pickle.load(f)
f.close()

targets_fp = os.path.join('..', 'superblue', 'superblue_18', 'targets.pkl')
f = open(targets_fp, 'rb')
targets_dict = pickle.load(f)
f.close()

In [4]:
num_instances = node_features_dict['num_instances']

In [5]:
len(metis_dict)

928383

In [6]:
partition_28 = []
for key, value in metis_dict.items():
    if value == 28 and key < num_instances:
        partition_28.append(key)

In [7]:
node_features_dict['instance_features'].shape

(459495, 6)

In [8]:
node_features_dict['instance_features'].max(axis=1)

array([234., 189., 232., ...,   6.,   6.,   6.])

In [9]:
node_features_dict['instance_features'].min(axis=1)

array([0., 0., 0., ..., 0., 0., 0.])

In [10]:
X = node_features_dict['instance_features'][partition_28]
y = targets_dict['demand'][partition_28]

In [11]:
feature_cols = [f"Feature_{i}" for i in range(X.shape[1])]
# Name for target
target_col = "Demand"

# Create DataFrame
df = pd.DataFrame(X, columns=feature_cols)
df[target_col] = y

In [12]:
df.shape

(2949, 7)

In [13]:
# Assume 'df' is our DataFrame with shape (2949, 7).
# Convert df to numpy array
data = df.values
data = (data - np.mean(data, axis=0)) / np.std(data, axis=0)

# By default, PC will treat all variables as continuous
# If the columns have different data types, some additional steps may be required.

# Run PC
cg = pc(
    data,
    alpha=0.01,  # Significance level for conditional independence tests
    indep_test='fisherz'  # Fisher's z test for continuous variables
)

Depth=0, working on node 2:  43%|████▎     | 3/7 [00:00<00:00, 245.92it/s] 

Depth=3, working on node 6: 100%|██████████| 7/7 [00:00<00:00, 1339.73it/s]


In [14]:
# cg is a CausalGraph object
# Print out the adjacency information (edges)
print("Edges (undirected before orientation):", cg.G.graph)

# Print the graph in a readable form
# GraphUtils.to_pydot(cg.G, labels=df.columns.tolist()).write("pc_graph.dot")

Edges (undirected before orientation): [[ 0 -1  0  0  0  0 -1]
 [ 1  0  0  1  0  0  1]
 [ 0  0  0 -1 -1  0  0]
 [ 0 -1  1  0  1  0  1]
 [ 0  0 -1 -1  0  0  0]
 [ 0  0  0  0  0  0  0]
 [-1 -1  0 -1  0  0  0]]


In [15]:
pyd = GraphUtils.to_pydot(cg.G, labels=df.columns.tolist())
pyd.write_png('pc_instance_features.png')

In [16]:
# data is the same df.values
ges_graph = ges(data)

# Print edges
# print("GES Edges:", ges_graph.G.graph)

# Export the graph in dot format
# GraphUtils.to_pydot(ges_graph.G, labels=df.columns.tolist()).write("ges_graph.dot")
pyd = GraphUtils.to_pydot(ges_graph['G'], labels=df.columns.tolist())
pyd.write_png('ges_instance_features.png')

In [17]:
ges_graph

{'update1': [[2, 4, ()],
  [3, 4, [2]],
  [3, 2, ()],
  [1, 3, (4,)],
  [1, 6, ()],
  [0, 6, [1]],
  [0, 2, ()],
  [1, 0, ()],
  [6, 3, ()],
  [6, 2, ()]],
 'update2': [],
 'G_step1': [<causallearn.graph.GeneralGraph.GeneralGraph at 0x30b8bc510>,
 'G_step2': [],
 'G': <causallearn.graph.GeneralGraph.GeneralGraph at 0x3120d8490>,
 'score': array([[-14003.82331939]])}